# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ayustar/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*


Finding 1: “The Anatomy of Growing Content” (a label question). The paper reports that pages with rising impressions are longer (about 3.2K vs 2.3K words) and younger (184 vs 230 days) than pages with falling impressions (74,187 rising vs 45,272 falling).

Question I would ask: The growing/declining label comes from comparing impressions in the last 30 days with the previous 30 days (±10%). The page features (average impressions, position) come from a rolling 90-day window that overlaps both of those periods. How much of the gap could come from how the label is built, and would it hold if features were taken from a window that ends before the label window starts?

Why it matters: With overlapping windows the finding describes what rising pages look like, not what makes a page grow. The paper’s own average impressions (2.8K vs 2.6K) and position (15.9 vs 16.1) are nearly equal across the two groups, which fits that reading.

What the paper does well: It calls the comparison observational, uses large samples, and says the gap is “directionally robust” without claiming causation.

Finding 2: ML appendix, “What Predicts Growth?” (a validation question). A logistic regression separates growing from declining pages with 71% holdout accuracy, using an 80/20 split of 61.8K active pages.

Question I would ask: Is the 80/20 split random or grouped by brand? With 57 brands, pages from the same brand could sit in both train and test. Would 71% hold if whole brands were held out? And what would a model that always predicts the majority class score? Finding 1’s class mix (74.8K up vs 45.6K down) puts that baseline near 62%, so 71% is a gain of about 9 points.

Why it matters: A random split can reward memorising brand-level patterns, and accuracy without a baseline is hard to read. Two of the strongest features, days visible and recent impressions, also sit close to the impressions-change label.

What the paper does well: It labels the ML section exploratory, says importances are descriptive, and says the direct comparisons come first.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

Row 1 is my Week-5 style setup. Row 2 keeps the same features but holds out whole clients. If row 2 drops, the random split was too easy. If both rows stay near-perfect, the features are the problem, which Section 3 checks

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.ensemble import RandomForestClassifier

if not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    !git clone https://github.com/Ayustar/flyrank-ml-internship-starter.git
    %cd flyrank-ml-internship-starter

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)]
df = df.drop_duplicates("content_id").reset_index(drop=True)
df["is_down"] = (df["trend_direction"] == "down").astype(int)

banned = ("trend", "label", "is_down", "score", "priority", "health", "flag", "action")
features = [c for c in df.select_dtypes("number").columns
            if c not in ("content_id", "client_id")
            and not any(b in c.lower() for b in banned)]

# Week-4 baseline rule (stale 90+ days and 500+ impressions), no label inputs
elig = (df["days_since_last_update"] >= 90) & (df["impressions_90d"] >= 500)
stale = (df["days_since_last_update"] / 180).clip(upper=1)
vis = np.log1p(df["impressions_90d"]) / np.log1p(df["impressions_90d"].max())
df["baseline_score"] = np.where(elig, 100 * (0.5 * stale + 0.5 * vis), 0)

def p_at_k(y, s, k):
    return y[np.argsort(-s, kind="stable")[:k]].mean()

def evaluate(tr, te, feats, label):
    Xtr, Xte = df.loc[tr, feats], df.loc[te, feats]
    ytr, yte = df.loc[tr, "is_down"].values, df.loc[te, "is_down"].values
    med = Xtr.median()
    rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=42)
    rf.fit(Xtr.fillna(med), ytr)
    s = rf.predict_proba(Xte.fillna(med))[:, 1]
    base = df.loc[te, "baseline_score"].values
    return {"setup": label, "n_test": len(te), "test decline rate": yte.mean(),
            "RF AUC": roc_auc_score(yte, s), "RF AP": average_precision_score(yte, s),
            "RF P@50": p_at_k(yte, s, 50),
            "baseline AUC": roc_auc_score(yte, base), "baseline P@50": p_at_k(yte, base, 50)}

idx = np.arange(len(df))
# BEFORE: random split (pages from one client can land on both sides)
tr_r, te_r = train_test_split(idx, test_size=0.2, stratify=df["is_down"], random_state=42)
# AFTER: client-holdout (test clients never seen in training)
tr_g, te_g = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
                  .split(df, df["is_down"], groups=df["client_id"]))
print("client overlap in client-holdout split:",
      len(set(df.loc[tr_g, "client_id"]) & set(df.loc[te_g, "client_id"])))

rows = [evaluate(tr_r, te_r, features, "1. random split, all features"),
        evaluate(tr_g, te_g, features, "2. client holdout, all features")]
print(pd.DataFrame(rows).set_index("setup").round(3))

Cloning into 'flyrank-ml-internship-starter'...
remote: Enumerating objects: 158, done.
remote: Counting objects: 100% (158/158), done.
remote: Compressing objects: 100% (114/114), done.
remote: Total 158 (delta 63), reused 92 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (158/158), 1.87 MiB | 3.84 MiB/s, done.
Resolving deltas: 100% (63/63), done.
/content/flyrank-ml-internship-starter
client overlap in client-holdout split: 0
                                 n_test  test decline rate  RF AUC  RF AP  \
setup                                                                       
1. random split, all features      6000              0.542   0.966  0.972   
2. client holdout, all features    6163              0.511   0.942  0.949   

                                 RF P@50  baseline AUC  baseline P@50  
setup                                                                  
1. random split, all features        1.0         0.531           0.48  
2. client holdout, all feature

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Leakage audit: I measured each feature’s single-column AUC on training clients and flagged any above 0.75 as suspects. Columns removed: (list them from the printout) because (one sentence each, for example “built from the same impressions-change window as the label”). The rest of the features are observable signals known before the decision point. trend_direction and is_down are never model inputs.

Failure examples: (2 to 3 sentences from the printed medians: what the wrong top-50 picks have in common, such as high impressions but stable traffic, and what a reviewer would check first.)

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
y_tr = df.loc[tr_g, "is_down"].values
aucs = {}
for c in features:
    x = df.loc[tr_g, c].fillna(df.loc[tr_g, c].median())
    a = roc_auc_score(y_tr, x)
    aucs[c] = max(a, 1 - a)
aucs = pd.Series(aucs).sort_values(ascending=False)
print(aucs.head(12).round(3))

suspects = list(aucs[aucs > 0.75].index)   # one column alone this strong is a red flag
print("\nsuspect leak columns:", suspects)
print("names to eyeball:", [c for c in features
      if any(w in c.lower() for w in ("change", "delta", "growth", "pct", "prev", "vs", "wow", "mom"))])

impressions_prev_30d      0.635
content_age_days          0.602
impressions_90d           0.600
age_tier_order            0.595
days_with_impressions     0.594
search_volume             0.570
sessions_prev_30d         0.552
days_since_last_update    0.545
avg_position              0.544
word_count                0.537
clicks_90d                0.537
sessions_last_30d         0.537
dtype: float64

suspect leak columns: []
names to eyeball: ['impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'ai_traffic_pct']


In [3]:
clean = [c for c in features if c not in suspects]
rows.append(evaluate(tr_g, te_g, clean, "3. client holdout, leak columns removed"))
final = pd.DataFrame(rows).set_index("setup").round(3)
print(final)
os.makedirs("work/outputs", exist_ok=True)
final.to_csv("work/outputs/w06_before_after.csv")

# real failure examples: wrong picks in the clean model's top 50 (numbers only, no identifiers)
med = df.loc[tr_g, clean].median()
rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=42)
rf.fit(df.loc[tr_g, clean].fillna(med), df.loc[tr_g, "is_down"])
te_df = df.loc[te_g].copy()
te_df["score"] = rf.predict_proba(te_df[clean].fillna(med))[:, 1]
top = te_df.sort_values("score", ascending=False).head(50)
wrong = top[top["is_down"] == 0]
print("\nwrong picks in top 50:", len(wrong))
show = ["score", "impressions_90d", "sessions_90d", "avg_position",
        "days_since_last_update", "content_age_days"]
print(wrong[show].head(8).round(2))
print("\nmedians, wrong picks vs all test pages:")
print(pd.DataFrame({"wrong picks": wrong[show[1:]].median(),
                    "all test pages": te_df[show[1:]].median()}).round(1))

                                         n_test  test decline rate  RF AUC  \
setup                                                                        
1. random split, all features              6000              0.542   0.966   
2. client holdout, all features            6163              0.511   0.942   
3. client holdout, leak columns removed    6163              0.511   0.942   

                                         RF AP  RF P@50  baseline AUC  \
setup                                                                   
1. random split, all features            0.972      1.0         0.531   
2. client holdout, all features          0.949      1.0         0.488   
3. client holdout, leak columns removed  0.949      1.0         0.488   

                                         baseline P@50  
setup                                                   
1. random split, all features                     0.48  
2. client holdout, all features                   0.30  
3. client holdo

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Copy the printed claims here as my final wording. Limits: the target is a current-window proxy (trend_direction == "down"), not a future outcome. The sample is a 30k-page starter slice. This is observational, and nothing here shows that a refresh causes recovery. All numbers are measured on one client-holdout split, so they are directional.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
r3 = final.iloc[2]; r1 = final.iloc[0]; r2 = final.iloc[1]
print(f"""ORIGINAL (Week 5): "Random Forest predicts declining pages with P@50 of 1.0."
REWRITE: Under a client-holdout split with {len(suspects)} leak-suspect column(s) removed, the Random Forest
ranked pages by an observed proxy for decline with AUC {r3['RF AUC']:.2f} and P@50 {r3['RF P@50']:.2f}
(n = {int(r3['n_test'])} test pages; random-pick precision {r3['test decline rate']:.2f}).
Under a random split with all features the same model scored AUC {r1['RF AUC']:.2f}, so the earlier
number overstated what carries to unseen clients. This is directional decision-support, not a forecast.

ORIGINAL (Week 4): "Staleness is confirmed as a decline signal."
REWRITE: Pages not updated in 90+ days declined more often in this sample (about 61% vs 51%). Across
held-out clients, the staleness-plus-visibility rule scored AUC {r3['baseline AUC']:.2f}, so the signal is
observed but weak as a ranking rule.""")

ORIGINAL (Week 5): "Random Forest predicts declining pages with P@50 of 1.0."
REWRITE: Under a client-holdout split with 0 leak-suspect column(s) removed, the Random Forest
ranked pages by an observed proxy for decline with AUC 0.94 and P@50 1.00
(n = 6163 test pages; random-pick precision 0.51).
Under a random split with all features the same model scored AUC 0.97, so the earlier
number overstated what carries to unseen clients. This is directional decision-support, not a forecast.

ORIGINAL (Week 4): "Staleness is confirmed as a decline signal."
REWRITE: Pages not updated in 90+ days declined more often in this sample (about 61% vs 51%). Across
held-out clients, the staleness-plus-visibility rule scored AUC 0.49, so the signal is
observed but weak as a ranking rule.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.